# 10.9 怎麼從向量方向比較圖文配對？


有兩張圖片卡和兩張文字卡，先只問「哪張說明卡最配這張圖」，不要求生成句子。下面用手設向量代表它們：第一對 `[1,0]` 與 `[2,0]` 同方向，第二對 `[0,1]` 與 `[0,3]` 同方向。

先除以自身長度，讓方向比較不受倍率支配，叫歸一化。例如 `[3,4]` 長度是 √(9+16)＝5，變成 `[0.6,0.8]`。歸一化向量的點積稱餘弦相似度，同方向 1、垂直 0、反方向 −1；這是數字空間的方向，並非圖上的左右。


In [1]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [2]:
import torch
from torch.nn import functional as F

image = torch.tensor([[1.0, 0.0], [0.0, 1.0]])
text = torch.tensor([[2.0, 0.0], [0.0, 3.0]])
similarity = F.normalize(image, dim=-1) @ F.normalize(text, dim=-1).T
print(similarity)
print("每張圖選文字欄", similarity.argmax(-1).tolist())

tensor([[1., 0.],
        [0., 1.]])
每張圖選文字欄 [0, 1]


`normalize(...,dim=-1)` 各自處理每列向量；文字矩陣轉置後，矩陣乘法算出 `[[1,0],[0,1]]`。每橫列是一張圖，每直欄是一段文字，選各列最大欄得到 `[0,1]`，與此例預先安排的配對一致。

相似度不是機率，一列不必加總為 1。`[1,1]` 歸一化後和 `[1,0]` 的相似度約 0.707；兩個候選可以同時得到這個值。之後若用分類代價學配對，才再將候選分數轉成機率。

這個手算核對了比較方式，沒有執行真實編碼器。真圖與真文字需要各自學出有用向量；同寬隨機特徵不會自動有配對意義。這種從候選找匹配內容的工作叫檢索，和寫出新的描述不同。

練習交換兩張文字卡，預期矩陣變成 `[[0,1],[1,0]]`、選中欄 `[1,0]`。配對身份不變，只有顯示位置改變；數值仍用浮點數，因為歸一化需要它。

<details>
<summary>回顧與查證</summary>

可回顧：[W.5向量與點積](https://birdhackor.github.io/tiny-perceptron-vlm/first-steps.html#W.5)、[10.5編碼器特徵](https://birdhackor.github.io/tiny-perceptron-vlm/10.5.html)。

原實驗的資料切分、更新設定與逐題結果見[既有實報](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/docs/course-experiments/results/contrastive.json)；重做入口見[實驗說明](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/docs/course-experiments/README.md)。這是上述有限任務的歷史紀錄。

</details>
